# IOAI — 2025 Contest Hogspell Challenge (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, glob, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
os.system('pip -q install -U diffusers accelerate')
if not os.path.exists('data/prompts.json'):
    os.environ['KAGGLE_API_TOKEN'] = 'KGAT_5dd261fded8e0d7eb2c29d8d65dfabea'  # 내장 토큰(규칙 수락된 계정)
    os.system('pip -q install kaggle')
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi(); api.authenticate()
    api.competition_download_files('neoai-2025-hogspell-challenge', path='data', quiet=False)
    for zp in glob.glob('data/*.zip'):
        with zipfile.ZipFile(zp) as zf: zf.extractall('data')
        os.remove(zp)
# 스티어링 코드(controller.py, steering_utils.py) — 레퍼런스에서 받고 not_steer 버그 보정
for f in ['controller.py', 'steering_utils.py']:
    if not os.path.exists(f):
        urllib.request.urlretrieve(f'https://raw.githubusercontent.com/lenjjiv/steering-hogs/main/{f}', f)
_s = open('steering_utils.py').read().replace('run_model(model_name, pipe, prompt, seed, num_denoising_steps, device)', 'run_model(pipe, prompt, seed, num_denoising_steps, device)')
open('steering_utils.py','w').write(_s)
print('데이터 준비:', 'data/prompts.json' if os.path.exists('data/prompts.json') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# NeoAI 2025 — The Hogspell Challenge

> **Northern Eurasia Olympiad in AI 2025 · Kaggle playground competition (디퓨전 · cross-attention 스티어링)**

요정 Metamorphosa의 저주처럼, **무슨 프롬프트가 들어와도 Stable Diffusion v1.5 가 돼지/멧돼지(hog) 이미지를
생성**하게 만드는 과제입니다. `prompts.json` 의 182개 프롬프트로 이미지를 생성해 `submission.csv`(`id,"0"`=PNG의
base64)로 제출합니다. **GPU 필요.**

**채점**: 정답(돼지 분류 점수)은 캐글 숨김 채점에만 있어, **Submissions** 탭에서 `submission.csv` 를 **캐글
리더보드에 자동 제출**해 점수를 받습니다(높을수록 좋음).

⚠️ **아래 베이스라인 = 스티어링 없는 vanilla SD 생성** → 프롬프트 그대로(기사·도시·말 등)라 돼지 점수 ≈ **0**.
UNet 의 cross-attention 을 "돼지" 방향으로 스티어링하면 ≈ 0.99 까지 오릅니다(모범답안 참고).
DGX 에서는 동봉 모델(`data/sd15`)을 쓰므로 다운로드가 필요 없습니다.

In [ ]:
import sys, os
for p in ['data', '.']:
    if p not in sys.path: sys.path.insert(0, p)
import torch, json, base64, io, pandas as pd, time
from PIL import Image
from diffusers import StableDiffusionPipeline
from steering_utils import compute_steering_vectors, generate_image
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device', dev)
# 모델: DGX 는 동봉 data/sd15(다운로드 불필요), Colab 등은 HF 에서 SD v1.5 다운로드
MODEL = 'data/sd15' if os.path.isdir('data/sd15') else 'sd-legacy/stable-diffusion-v1-5'
pipe = StableDiffusionPipeline.from_pretrained(MODEL, torch_dtype=torch.float16, variant='fp16',
            safety_checker=None, requires_safety_checker=False).to(dev)
pipe.set_progress_bar_config(disable=True)
prompts = json.load(open('data/prompts.json' if os.path.exists('data/prompts.json') else 'prompts.json'))
def to_b64(img):
    buf = io.BytesIO(); img.save(buf, format='PNG'); return base64.b64encode(buf.getvalue()).decode()
print('prompts', len(prompts), '| model', MODEL)

In [ ]:
# BASELINE: 스티어링 없이 그대로 생성
t = time.time(); ids, b64s = [], []
for k, p in prompts.items():
    img = generate_image(pipe, p, seed=42, not_steer=True, num_denoising_steps=25)
    ids.append(k); b64s.append(to_b64(img))
pd.DataFrame({'id': ids, '0': b64s}).to_csv('submission.csv', index=False)
print('wrote submission.csv', len(ids), round(time.time()-t,1),'s — 스티어링으로 돼지화하세요(모범답안)')


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)